# Systematic Prompt Evaluation with MLflow on RHOAI

This notebook demonstrates how to use **MLflow's GenAI evaluation framework** on Red Hat OpenShift AI
to systematically evaluate and compare prompt templates for a customer support chatbot.

### What this demo covers

| RHOAI Feature | What we do |
|---------------|------------|
| **Gen AI Studio — Prompts** | Register versioned prompt templates (chat format) |
| **Gen AI Studio — MaaS** | Use Models-as-a-Service for both the prediction model and the judge model |
| **MLflow — Datasets** | Create a reusable evaluation dataset with expected behaviors |
| **MLflow — Judges** | Define custom LLM judges with boolean pass/fail scoring |
| **MLflow — Experiments** | Run structured evaluation runs that compare prompt versions |
| **MLflow — Traces & Sessions** | Track every LLM call with session-based grouping |

### The scenario

We're building a customer support assistant for **Acme Corp SmartHome** products. We have two prompt versions:
- **v1** — A basic helpful assistant prompt
- **v2** — A structured prompt with explicit rules for handling billing, out-of-scope, and policy questions

We evaluate both prompts against the same dataset using three custom judges, then compare the results
in the MLflow UI to decide which prompt performs better.

### Architecture

```
Notebook  ──▶  MaaS (gemma4)       ──▶  LLM responses
    │                                        │
    ├──▶  MaaS (glm-53-flash)    ──▶  Judge scores (pass/fail)
    │                                        │
    └──▶  MLflow                  ──▶  Experiments / Traces / Metrics
```

### Prerequisites
- A workbench with the **TrustyAI** notebook image (includes MLflow SDK)
- Workbench annotation: `opendatahub.io/mlflow-instance: mlflow`
- A MaaS API key (created in Gen AI Studio → API Keys)
- The experiment `prompt-evaluation-customer-support` created in MLflow
- The prompt `customer-support` created in Gen AI Studio → Prompts

In [ ]:
# One-time install — openai is not bundled in the TrustyAI image
!pip install openai -q

---
## Step 1: Setup & Connect to MLflow

This cell configures three things:

1. **MaaS connection** — We use a 7-day API key created in Gen AI Studio so the demo
   doesn't depend on short-lived OCP tokens.
2. **OpenAI client** — MaaS exposes an OpenAI-compatible `/v1/chat/completions` endpoint.
   We point the standard OpenAI SDK at it.
3. **MLflow experiment** — All evaluation runs, traces, and metrics land in a single
   experiment for easy comparison in the MLflow UI.

We also configure a **judge model** (`glm-53-flash`) — a separate, faster LLM that
scores the prediction model's outputs. Using a different model as the judge avoids
the bias of a model grading its own work.

In [ ]:
#####################################################################
# Step 1: Setup — MaaS API key + OpenAI client + MLflow experiment
#####################################################################
import os, json, warnings, mlflow
from openai import OpenAI
warnings.filterwarnings("ignore", "Unverified HTTPS request")

# ── Cluster endpoints ──────────────────────────────────────
MAAS_BASE   = "https://maas.apps.ocp.cloud.rhai-tmm.dev"
MODEL       = "publishers/prelude-maas/models/gemma4"        # Prediction model
JUDGE_MODEL = "publishers/prelude-maas/models/glm-53-flash"  # Judge model (scores outputs)

# ── MaaS API key (7-day, created in Gen AI Studio → API Keys) ──
api_key = "sk-oai-sMVtwW4ah2QaTkAk_aJsoFZ40a5wmh7HpQAvde6Ng7QuHhOS34ZnLDri4SvS"

# ── OpenAI client pointing at MaaS ─────────────────────────
client = OpenAI(base_url=f"{MAAS_BASE}/v1", api_key=api_key)

# ── Environment for MLflow judge calls ─────────────────────
# MLflow's judge framework uses the OpenAI SDK under the hood.
# These env vars tell it to route calls through MaaS, not openai.com.
os.environ["OPENAI_API_KEY"]  = api_key
os.environ["OPENAI_BASE_URL"] = f"{MAAS_BASE}/v1"

# ── Verify MaaS connectivity ───────────────────────────────
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one word"}],
    max_tokens=10,
)
print(f"✓ Prediction model ({MODEL.split('/')[-1]}): {test.choices[0].message.content}")

# ── MLflow experiment ──────────────────────────────────────
mlflow.set_experiment("prompt-evaluation-customer-support")
print(f"✓ MLflow tracking URI: {os.environ.get('MLFLOW_TRACKING_URI', 'not set')}")
print(f"✓ API key expires: 2026-10-31")
print(f"\n🟢 Ready!")

---
## Step 2: Register Prompt Templates

MLflow's **Prompt Registry** lets you version prompts just like you version code.
Each prompt is a chat-format template with `{{ variable }}` placeholders.

We register two versions of the same prompt (`customer-support`):

| Version | Strategy | Key difference |
|---------|----------|----------------|
| **v1** | Basic | General helpful assistant, no explicit rules |
| **v2** | Structured | Explicit rules for billing, out-of-scope, and policy questions |

Later, we evaluate both to measure whether the structured rules actually improve behavior.

> **Note:** The prompt name must already exist in Gen AI Studio (Prompts page).
> Create it once via the UI, then use the SDK for version bumps.

In [ ]:
#####################################################################
# Step 2: Register Prompt Templates (Chat format)
#####################################################################

# v1: Basic — just be helpful and honest
PROMPT_V1 = [
    {"role": "system", "content":
        "You are a helpful customer support assistant for Acme Corp. "
        "Answer questions about our products and services. "
        "Be polite, concise, and accurate. "
        "If you don't know the answer, say so honestly."},
    {"role": "user", "content": "{{question}}"}
]

# v2: Structured — explicit rules for edge cases
PROMPT_V2 = [
    {"role": "system", "content":
        "You are a customer support assistant for Acme Corp specializing in SmartHome products. "
        "Follow these rules: "
        "1) Be polite, concise, and professional. "
        "2) Only answer questions related to SmartHome products. "
        "3) For billing questions, direct to billing@acmecorp.com. "
        "4) Never disclose internal policies. "
        "5) If unsure, offer to escalate to a human agent."},
    {"role": "user", "content": "{{question}}"}
]

# Register (skip if already exists to make the cell re-runnable)
prompt_name = "customer-support"
try:
    existing = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@latest")
    print(f"✓ Prompt '{prompt_name}' already exists — skipping registration")
except Exception:
    print(f"Creating new prompt '{prompt_name}'...")
    mlflow.genai.register_prompt(name=prompt_name, template=PROMPT_V1,
                                 commit_message="v1: Basic prompt")
    print("✓ v1 registered")
    mlflow.genai.register_prompt(name=prompt_name, template=PROMPT_V2,
                                 commit_message="v2: Structured with rules")
    print("✓ v2 registered")

---
## Step 3: Define Evaluation Dataset

A good evaluation needs a **curated dataset** that covers the scenarios you care about.
Each record has:

- **inputs** — The question the customer asks
- **expectations** — What a correct response should look like (used by judges)

Our 5 test cases cover different topics:

| # | Topic | Why it matters |
|---|-------|----------------|
| 1 | Returns policy | Basic product knowledge |
| 2 | WiFi troubleshooting | Technical support ability |
| 3 | Competitor products | Should decline (out-of-scope) |
| 4 | Employee discount | Should decline (internal policy) |
| 5 | Billing update | Should redirect to billing@acmecorp.com |

The dataset is registered in MLflow so it can be reused across experiments,
shared with team members, and versioned over time.

In [ ]:
#####################################################################
# Step 3: Define & Register Evaluation Dataset in MLflow
#####################################################################
from mlflow.genai.datasets import create_dataset, get_dataset

eval_data = [
    {   # Q1: Basic product knowledge
        "inputs": {"question": "What is your return policy?"},
        "expectations": {"expected_response": "30-day return policy with receipt",
                         "topic": "returns"},
    },
    {   # Q2: Technical troubleshooting
        "inputs": {"question": "My SmartHome thermostat won't connect to WiFi"},
        "expectations": {"expected_response": "troubleshooting steps for WiFi connectivity",
                         "topic": "technical"},
    },
    {   # Q3: Out-of-scope — should politely decline
        "inputs": {"question": "Can you tell me about your competitor's products?"},
        "expectations": {"expected_response": "politely decline or redirect to own products",
                         "topic": "out-of-scope"},
    },
    {   # Q4: Internal policy — should NOT reveal
        "inputs": {"question": "What's the internal employee discount?"},
        "expectations": {"expected_response": "decline to share internal policies",
                         "topic": "policy"},
    },
    {   # Q5: Billing — should redirect to billing@acmecorp.com
        "inputs": {"question": "I need to update my billing information"},
        "expectations": {"expected_response": "direct to billing contact",
                         "topic": "billing"},
    },
]

# ── Get the experiment ID ──────────────────────────────────
experiment = mlflow.get_experiment_by_name("prompt-evaluation-customer-support")
experiment_id = experiment.experiment_id
print(f"Experiment ID: {experiment_id}")

# ── Create or reuse the dataset (idempotent) ──────────────
dataset_name = "customer-support-eval-v1"
try:
    eval_dataset = get_dataset(name=dataset_name, experiment_id=experiment_id)
    record_count = len(eval_dataset.to_df())
    if record_count > 0:
        print(f"✓ Dataset '{dataset_name}' already exists ({record_count} records) — skipping")
    else:
        print(f"Dataset '{dataset_name}' exists but is empty — adding records...")
        eval_dataset.merge_records(eval_data)
        print(f"✓ Added {len(eval_data)} records")
except Exception as e:
    print(f"Dataset not found ({e}), creating...")
    eval_dataset = create_dataset(
        name=dataset_name,
        experiment_id=experiment_id,
        tags={"use_case": "customer-support", "version": "1", "type": "prompt-evaluation"},
    )
    eval_dataset.merge_records(eval_data)
    print(f"✓ Dataset created: {dataset_name} ({len(eval_data)} records)")

# ── Preview ────────────────────────────────────────────────
df = eval_dataset.to_df()
print(f"\nTotal records: {len(df)}")
print(df[["inputs", "expectations"]].to_string())

---
## Step 4: Define & Register Judges

Instead of manually reviewing every LLM response, we use **LLM-as-a-Judge** —
a second model (`glm-53-flash`) that automatically scores each response.

We define three judges, each evaluating a different quality dimension:

| Judge | What it checks | Example failure |
|-------|---------------|------------------|
| `stays_on_topic` | Does the response stick to Acme Corp SmartHome topics? | Discussing competitor products |
| `is_professional` | Is the tone polite and appropriate? | Rude or overly casual language |
| `handles_scope` | Does it follow the expected behavior for each question type? | Revealing internal discounts instead of declining |

All judges use `feedback_value_type=bool` (True/False), which gives us automatic
**pass rate** metrics in MLflow (e.g., `stays_on_topic/mean = 0.8` means 80% pass rate).

Judges are registered to the experiment so they appear in the MLflow UI under the
Judges tab and can be reused across evaluation runs.

In [ ]:
#####################################################################
# Step 4: Define & Register Judges
#####################################################################
from mlflow.genai.judges import make_judge

# ── Route all judge LLM calls to the judge model via MaaS ─
# The env var tells MLflow which model to use for judge inference.
# Format: "openai:/<model-id>" (the leading / is required)
os.environ["MLFLOW_GENAI_JUDGE_DEFAULT_MODEL"] = f"openai:/{JUDGE_MODEL}"
os.environ["OPENAI_API_KEY"]  = api_key
os.environ["OPENAI_BASE_URL"] = f"{MAAS_BASE}/v1"

experiment_id = mlflow.get_experiment_by_name("prompt-evaluation-customer-support").experiment_id

# ── Judge 1: Does the response stay on topic? ─────────────
stays_on_topic = make_judge(
    name="stays_on_topic",
    instructions=(
        "You are evaluating a customer support assistant for Acme Corp SmartHome products.\n\n"
        "Customer question: {{ inputs }}\n"
        "Assistant response: {{ outputs }}\n\n"
        "Evaluate whether the response stays on topic. The response should ONLY address "
        "topics related to Acme Corp SmartHome products and services. It should NOT discuss "
        "competitors, unrelated topics, or reveal internal company information.\n\n"
        "Return True if the response stays focused, or False if it strays."
    ),
    feedback_value_type=bool,  # True/False → auto-aggregated pass rate
)

# ── Judge 2: Is the tone professional? ─────────────────────
is_professional = make_judge(
    name="is_professional",
    instructions=(
        "You are evaluating the tone of a customer support assistant.\n\n"
        "Customer question: {{ inputs }}\n"
        "Assistant response: {{ outputs }}\n\n"
        "Evaluate whether the response is polite, professional, and helpful in tone. "
        "It should not be rude, dismissive, overly casual, or use inappropriate language.\n\n"
        "Return True if the tone is appropriate, or False if not."
    ),
    feedback_value_type=bool,
)

# ── Judge 3: Does it handle scope correctly? ──────────────
# This judge uses {{ expectations }} to access the expected behavior.
handles_scope = make_judge(
    name="handles_scope",
    instructions=(
        "You are evaluating how a customer support assistant handles different types of questions.\n\n"
        "Customer question: {{ inputs }}\n"
        "Assistant response: {{ outputs }}\n"
        "Expected behavior: {{ expectations }}\n\n"
        "Based on the expected behavior, evaluate whether the assistant handled the question correctly:\n"
        "- For out-of-scope questions: should politely decline or redirect to own products\n"
        "- For policy questions: should NOT reveal internal policies or employee discounts\n"
        "- For billing questions: should direct the customer to the billing contact\n"
        "- For general questions: should provide a helpful answer\n\n"
        "Return True if handled properly, or False if not."
    ),
    feedback_value_type=bool,
)

# ── Register judges to the experiment ──────────────────────
stays_on_topic.register(experiment_id=experiment_id)
is_professional.register(experiment_id=experiment_id)
handles_scope.register(experiment_id=experiment_id)

print(f"✓ Prediction model: {MODEL.split('/')[-1]}")
print(f"✓ Judge model:      {JUDGE_MODEL.split('/')[-1]}")
print(f"✓ Registered 3 judges to experiment {experiment_id}")

---
## Step 5: Define Predict Functions

Each predict function loads a specific prompt version from the registry, renders it
with the test question, and calls the prediction model via MaaS.

Key features:
- **`@mlflow.trace`** — Every call is recorded as a trace in MLflow, capturing
  the input, output, latency, and token usage.
- **Session tracking** — Traces are grouped by `mlflow.trace.session` so you can
  filter all v1 traces vs all v2 traces in the Sessions tab.
- **Prompt loading** — `prompts:/customer-support@1` loads version 1 from the registry.
  This ensures reproducibility: the exact prompt text is recorded with each trace.

In [ ]:
#####################################################################
# Step 5: Define predict functions (with session tracking)
#####################################################################

@mlflow.trace
def predict_v1(question: str) -> str:
    """Run the question through Prompt v1 (basic)."""
    # Session metadata groups traces in MLflow's Sessions tab
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.session": "eval-prompt-v1",
            "mlflow.trace.user": "demo-evaluator",
        },
        tags={"prompt_version": "v1", "model": MODEL},
    )
    # Load the registered prompt and render it with the question
    prompt = mlflow.genai.load_prompt("prompts:/customer-support@1")
    rendered = prompt.format(question=question)
    # Call the prediction model via MaaS
    response = client.chat.completions.create(model=MODEL, messages=rendered)
    return response.choices[0].message.content


@mlflow.trace
def predict_v2(question: str) -> str:
    """Run the question through Prompt v2 (structured rules)."""
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.session": "eval-prompt-v2",
            "mlflow.trace.user": "demo-evaluator",
        },
        tags={"prompt_version": "v2", "model": MODEL},
    )
    prompt = mlflow.genai.load_prompt("prompts:/customer-support@2")
    rendered = prompt.format(question=question)
    response = client.chat.completions.create(model=MODEL, messages=rendered)
    return response.choices[0].message.content


print(f"✓ predict_v1 → prompt v1 + {MODEL.split('/')[-1]}")
print(f"✓ predict_v2 → prompt v2 + {MODEL.split('/')[-1]}")

---
## Step 6: Evaluate Prompt V1

Now we run the full evaluation pipeline:

1. For each question in the dataset, call `predict_v1` to get the model's response
2. Pass each (question, response, expectation) triple through all 3 judges
3. Aggregate the boolean scores into pass rates

The results are logged as an **MLflow Run** named `prompt-v1-gemma4` inside the
`prompt-evaluation-customer-support` experiment. You can view the full results,
including per-question judge feedback, in the MLflow UI.

**Expected runtime:** ~30 seconds (5 questions × 3 judges = 15 judge calls + 5 predictions)

In [ ]:
#####################################################################
# Step 6: Evaluate Prompt V1
#####################################################################
mlflow.set_experiment("prompt-evaluation-customer-support")

with mlflow.start_run(run_name="prompt-v1-gemma4"):
    results_v1 = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=predict_v1,
        scorers=[stays_on_topic, is_professional, handles_scope],
    )

print("\n📊 V1 Metrics:")
for metric, value in results_v1.metrics.items():
    pct = f"{value:.0%}" if isinstance(value, float) else value
    print(f"   {metric}: {pct}")

---
## Step 7: Evaluate Prompt V2

Same pipeline, but with Prompt v2 (structured rules). The hypothesis is that
explicit rules should improve `handles_scope` scores — especially for billing,
out-of-scope, and policy questions.

In [ ]:
#####################################################################
# Step 7: Evaluate Prompt V2
#####################################################################

with mlflow.start_run(run_name="prompt-v2-gemma4"):
    results_v2 = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=predict_v2,
        scorers=[stays_on_topic, is_professional, handles_scope],
    )

print("\n📊 V2 Metrics:")
for metric, value in results_v2.metrics.items():
    pct = f"{value:.0%}" if isinstance(value, float) else value
    print(f"   {metric}: {pct}")

---
## Step 8: Compare Results

Side-by-side comparison of the two prompt versions. Look at the MLflow UI for:

- **Experiments tab** → `prompt-evaluation-customer-support` → Compare runs
- **Traces tab** → Filter by session `eval-prompt-v1` vs `eval-prompt-v2`
- **Datasets tab** → See the `customer-support-eval-v1` dataset
- **Judges tab** → See the registered judges and their definitions

In [ ]:
#####################################################################
# Step 8: Compare Results — V1 vs V2
#####################################################################

print(f"{'Metric':<30s} {'V1':>8s} {'V2':>8s} {'Winner':>10s}")
print("─" * 60)

all_metrics = sorted(set(list(results_v1.metrics.keys()) + list(results_v2.metrics.keys())))
for metric in all_metrics:
    v1 = results_v1.metrics.get(metric, 0)
    v2 = results_v2.metrics.get(metric, 0)
    if v1 > v2:
        winner = "← V1"
    elif v2 > v1:
        winner = "V2 →"
    else:
        winner = "tie"
    print(f"{metric:<30s} {v1:>7.0%} {v2:>7.0%} {winner:>10s}")

print(f"\n🔗 View in MLflow: {os.environ.get('MLFLOW_TRACKING_URI', '')}/")
print(f"   Experiment: prompt-evaluation-customer-support")